# Notebook 02 — Target Preparation

**Goals of this notebook:**
1. Download 1M17 (EGFR WT) and 4I22 (EGFR T790M) from RCSB PDB
2. Identify the co-crystal ligand residue names
3. Extract the binding pocket (all residues within 10 Å of ligand center)
4. Generate 3D conformers for the three baseline drugs (Erlotinib, Gefitinib, Osimertinib)
5. Visualize the WT pocket with py3Dmol

**Inputs:** nothing (downloads directly from RCSB + uses SMILES from `configs/targets.yaml`)

**Outputs saved to Drive:**
- `data/pdb/1M17.pdb`, `data/pdb/4I22.pdb`
- `data/pockets/1M17_pocket.pdb`, `data/pockets/4I22_pocket.pdb`
- `data/pocket_centers.json` (ligand centers used for Vina box placement)
- `data/baselines/erlotinib.sdf`, `gefitinib.sdf`, `osimertinib.sdf`

Runtime: **CPU** is fine. Total execution: ~2 minutes.


In [ ]:
# ── Colab bootstrap (identical across all notebooks) ────────────────────────
REPO_URL = "https://github.com/Jonathan-Ye-1/egfr-drug-design-eecs6895-final-project.git"

import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
if not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.colab_init import setup
PROJECT_ROOT = setup(repo_url=REPO_URL)


In [ ]:
# ── Download PDB files ──────────────────────────────────────────────────────
def download_pdb(pdb_id: str, out_dir: str) -> str:
    out_path = os.path.join(out_dir, f'{pdb_id}.pdb')
    if os.path.exists(out_path):
        print(f'{pdb_id}.pdb already downloaded.')
        return out_path
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    r = requests.get(url)
    r.raise_for_status()
    with open(out_path, 'w') as f:
        f.write(r.text)
    print(f'Downloaded {pdb_id}.pdb ({len(r.text)//1024} KB)')
    return out_path

pdb_dir = f'{PROJECT_ROOT}/data/pdb'
p_1m17 = download_pdb('1M17', pdb_dir)
p_4i22 = download_pdb('4I22', pdb_dir)

In [ ]:
# ── Identify ligand residue names ───────────────────────────────────────────
# Run this to verify ligand resnames in case they differ from config
def get_hetatm_resnames(pdb_file):
    names = set()
    with open(pdb_file) as f:
        for line in f:
            if line.startswith('HETATM'):
                names.add(line[17:20].strip())
    return names

print('1M17 HETATM residues:', get_hetatm_resnames(p_1m17))
print('4I22 HETATM residues:', get_hetatm_resnames(p_4i22))

In [ ]:
# ── Extract pockets ─────────────────────────────────────────────────────────
from src.utils import load_config
from src.pocket_extraction import extract_pocket
import numpy as np

cfg = load_config(f'{PROJECT_ROOT}/configs/targets.yaml')

wt_cfg  = cfg['targets']['wildtype']
mut_cfg = cfg['targets']['mutant']

center_wt  = extract_pocket(p_1m17, wt_cfg['ligand_resname'],
                             f'{PROJECT_ROOT}/data/pockets/1M17_pocket.pdb')
center_mut = extract_pocket(p_4i22, mut_cfg['ligand_resname'],
                             f'{PROJECT_ROOT}/data/pockets/4I22_pocket.pdb')

print('WT  center:', center_wt.round(2))
print('Mut center:', center_mut.round(2))

# Save centers for Vina notebook
import json
centers = {'1M17': center_wt.tolist(), '4I22': center_mut.tolist()}
with open(f'{PROJECT_ROOT}/data/pocket_centers.json', 'w') as f:
    json.dump(centers, f, indent=2)
print('Centers saved → data/pocket_centers.json')


In [ ]:
# ── Generate baseline drug 3D conformers ────────────────────────────────────
from rdkit import Chem
from rdkit.Chem import AllChem

baselines = cfg['baselines']
for name, info in baselines.items():
    sdf_path = os.path.join(PROJECT_ROOT, info['sdf_file'])
    mol = Chem.MolFromSmiles(info['smiles'])
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)
    w = Chem.SDWriter(sdf_path)
    w.write(mol)
    w.close()
    print(f'{info["name"]} conformer saved → {info["sdf_file"]}')

In [ ]:
# ── Visualize WT pocket with py3Dmol ────────────────────────────────────────
import py3Dmol

with open(f'{PROJECT_ROOT}/data/pockets/1M17_pocket.pdb') as f:
    pdb_str = f.read()

view = py3Dmol.view(width=700, height=500)
view.addModel(pdb_str, 'pdb')
view.setStyle({'cartoon': {'color': 'lightblue'}})
view.setStyle({'resn': wt_cfg['ligand_resname']}, {'stick': {'colorscheme': 'greenCarbon'}})
view.zoomTo()
view.show()